<h1 style="color:#1B4F72; font-weight:bold;">
Feature Engineering for Hierarchical Gas Classification
</h1>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This notebook continues the OBSeRVeD pipeline from the Data Understanding and Preprocessing stages. 
Its purpose is to transform preprocessed windowed time-series sensor data into informative features 
for machine learning-based gas pattern recognition.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Feature extraction focuses on temporal signal characteristics, including trends, variability, transitions, 
and response–recovery dynamics. Since the dataset is inherently time-series, the goal is to capture 
meaningful temporal patterns within each fixed-length window. <i>(Literature Review, Section 3, Segmentation & Feature Engineering)</i>
</p>

<h2 style="color:#1B4F72; font-weight:bold;">
1. Feature Engineering Overview
</h2>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This stage converts windowed sensor signals into a structured feature matrix for modeling. 
Features are organized into statistical, temporal, shape-based, and variability groups, followed by aggregation 
into a final dataset suitable for machine learning. <i>(Literature Review, Section 3.2, Feature Engineering for Gas Sensor Time-Series)</i>
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
Data Input
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The preprocessed fixed-window dataset from the previous stage is loaded as the input for all feature extraction steps. <i>(Literature Review, Section 3.1, Segmentation and Windowing Strategies in Gas Sensor Time-Series)</i>
</p>

In [8]:
from pathlib import Path
import pandas as pd

# Path to processed data
DATA_PATH = Path("../data/processed")

# Load preprocessed window-level dataset
windowed_df = pd.read_parquet(DATA_PATH / "windowed_timeseries.parquet")

print("Windowed dataset shape:", windowed_df.shape)
display(windowed_df.head())

Windowed dataset shape: (2353, 11)


,run_id,experiment,experiment_folder,run_folder,repeat_index,window_id,start_idx,end_idx,time_start,time_end,signal_window
0,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,NaN,Mixing_LowHigh_concentrations__UV ink-exposed ...,0,99,0.000,115.869,"[0.22191272881048008, 0.1788521326708596, 0.14..."
1,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,NaN,Mixing_LowHigh_concentrations__UV ink-exposed ...,50,149,58.252,175.526,"[0.10882198394731375, 0.10867808536671446, 0.1..."
2,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,NaN,Mixing_LowHigh_concentrations__UV ink-exposed ...,100,199,117.016,234.546,"[0.03248668794186581, 0.03501793439588855, 0.0..."
3,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,NaN,Mixing_LowHigh_concentrations__UV ink-exposed ...,150,249,176.640,293.550,"[0.08211229917865742, 0.07837054760784211, 0.0..."
4,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,NaN,Mixing_LowHigh_concentrations__UV ink-exposed ...,200,299,235.821,352.716,"[0.09998657123741517, 0.10686870133679063, 0.1..."


<h2 style="color:#1B4F72; font-weight:bold;">
2. Statistical Features
</h2>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Statistical features provide compact summaries of the signal distribution within each window. 
These descriptors capture central tendency (mean, median), dispersion (standard deviation, interquartile range), 
extremes (minimum, maximum, range), and relative variability (coefficient of variation). 
Together, they offer a stable and noise-robust representation of signal magnitude and variability, 
serving as a foundational feature group for downstream modeling. <i>(Literature Review, Section 3.2, Feature Engineering for Gas Sensor Time-Series)</i>
</p>

In [9]:
import numpy as np
import pandas as pd

def extract_stat_features(signal):
    signal = np.asarray(signal)

    mean_val = np.mean(signal)
    std_val = np.std(signal)

    q25 = np.percentile(signal, 25)
    q75 = np.percentile(signal, 75)

    return {
        "mean": mean_val,
        "std": std_val,
        "min": np.min(signal),
        "max": np.max(signal),
        "median": np.median(signal),
        "range": np.max(signal) - np.min(signal),
        "iqr": q75 - q25,
        "cv": std_val / mean_val if mean_val != 0 else 0
    }

stat_features = windowed_df["signal_window"].apply(extract_stat_features)
stat_df = pd.DataFrame(stat_features.tolist())

display(stat_df.head())

,mean,std,min,max,median,range,iqr,cv
0,0.089142,0.031639,0.020605,0.221913,0.094835,0.201308,0.033844,0.354935
1,0.082972,0.028013,0.020605,0.128674,0.089349,0.108069,0.035378,0.337617
2,0.080224,0.019554,0.030791,0.113978,0.082813,0.083187,0.023622,0.243743
3,0.093657,0.021975,0.057616,0.147132,0.090766,0.089516,0.033182,0.234636
4,0.088229,0.035386,0.008150,0.147132,0.091891,0.138982,0.048667,0.401074


<h2 style="color:#1B4F72; font-weight:bold;">
3. Temporal Features
</h2>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Temporal features characterize the dynamic behavior of the signal within each window by analyzing first-order differences. 
These features capture local transitions, response intensity, and directional changes, providing insight into how the signal evolves over time rather than just its static distribution. <i>(Literature Review, Section 3.2, Feature Engineering for Gas Sensor Time-Series)</i>
</p>

In [11]:
import numpy as np

def extract_temporal_features(signal):
    signal = np.asarray(signal)
    diff = np.diff(signal)

    # Handle short signals safely
    if len(signal) < 3:
        return {
            "first_diff_mean": 0,
            "first_diff_std": 0,
            "first_diff_max": 0,
            "first_diff_min": 0,
            "abs_diff_mean": 0,
            "diff_sign_changes": 0,
            "slope": 0,
            "second_diff_mean": 0
        }

    # First-order features
    first_diff_mean = np.mean(diff)
    first_diff_std = np.std(diff)
    first_diff_max = np.max(diff)
    first_diff_min = np.min(diff)
    abs_diff_mean = np.mean(np.abs(diff))

    # Normalized sign changes (robust)
    sign_changes = np.sum(np.diff(np.sign(diff)) != 0)
    sign_changes_norm = sign_changes / len(diff)

    # Trend (slope)
    x = np.arange(len(signal))
    slope = np.polyfit(x, signal, 1)[0]

    # Second-order difference (acceleration)
    second_diff = np.diff(diff)
    second_diff_mean = np.mean(second_diff) if len(second_diff) > 0 else 0

    return {
        "first_diff_mean": first_diff_mean,
        "first_diff_std": first_diff_std,
        "first_diff_max": first_diff_max,
        "first_diff_min": first_diff_min,
        "abs_diff_mean": abs_diff_mean,
        "diff_sign_changes": sign_changes_norm,
        "slope": slope,
        "second_diff_mean": second_diff_mean
    }


temporal_features = windowed_df["signal_window"].apply(extract_temporal_features)
temporal_df = pd.DataFrame(temporal_features.tolist())

display(temporal_df.head())

,first_diff_mean,first_diff_std,first_diff_max,first_diff_min,abs_diff_mean,diff_sign_changes,slope,second_diff_mean
0,-0.001952,0.008036,0.011225,-0.043061,0.004951,0.191919,-0.000115,0.000487
1,-0.000255,0.004661,0.012690,-0.010431,0.003665,0.222222,-0.000426,-0.000062
2,0.000605,0.004608,0.012690,-0.008022,0.003636,0.222222,0.000232,0.000021
3,0.000504,0.004457,0.010323,-0.009708,0.003719,0.161616,0.000182,-0.000061
4,-0.000655,0.005779,0.011907,-0.012680,0.004929,0.121212,-0.000560,-0.000023


<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Temporal features provide insight into how the signal evolves within each window rather than simply summarizing its distribution. 
The mean of first differences is generally close to zero, indicating that increases and decreases tend to balance over short intervals, which is expected in cyclic exposure–recovery patterns.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The standard deviation of differences varies across windows, reflecting differences in transition intensity. 
Higher values correspond to sharper response or recovery phases, while lower values indicate relatively stable regions of the signal.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The mean absolute difference captures overall activity regardless of direction, highlighting segments with stronger temporal variation. 
In contrast, the normalized number of sign changes reflects how frequently the signal direction fluctuates, providing an indication of oscillatory or noisy behavior.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The slope feature summarizes the overall trend within each window, distinguishing between increasing, decreasing, or stable segments. 
Additionally, second-order differences provide a coarse measure of acceleration, capturing how rapidly transitions themselves change over time.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, temporal features complement statistical descriptors by encoding dynamic response characteristics, which are critical for distinguishing gas exposure patterns and sensor behavior across conditions.
</p>

<h2 style="color:#1B4F72; font-weight:bold;">
4. Shape-Based Features
</h2>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Shape-based features describe the overall morphology of the signal within each window, focusing on global structure rather than local fluctuations. 
These features capture trend direction, peak characteristics, and response–recovery behavior, which are critical for distinguishing gas-specific signal patterns. <i>(Literature Review, Section 3.2, Feature Engineering for Gas Sensor Time-Series)</i>
</p>

In [13]:
import numpy as np
from scipy.stats import linregress

def extract_shape_features(signal):
    signal = np.asarray(signal)
    n = len(signal)

    if n < 3:
        return {
            "trend_slope": 0,
            "trend_type": 0,
            "peak_value": 0,
            "peak_position": 0,
            "rise_strength": 0,
            "fall_strength": 0
        }

    x = np.arange(n)

    # Trend
    slope, _, _, _, _ = linregress(x, signal)
    trend_threshold = 1e-3

    if slope > trend_threshold:
        trend_type = 1
    elif slope < -trend_threshold:
        trend_type = -1
    else:
        trend_type = 0

    # Peak
    peak_value = np.max(signal)
    peak_idx = np.argmax(signal)
    peak_position = peak_idx / n  # normalized

    # Rise / Fall
    rise_strength = peak_value - signal[0]
    fall_strength = peak_value - signal[-1]

    return {
        "trend_slope": slope,
        "trend_type": trend_type,
        "peak_value": peak_value,
        "peak_position": peak_position,
        "rise_strength": rise_strength,
        "fall_strength": fall_strength
    }


shape_features = windowed_df["signal_window"].apply(extract_shape_features)
shape_df = pd.DataFrame(shape_features.tolist())

display(shape_df.head())

,trend_slope,trend_type,peak_value,peak_position,rise_strength,fall_strength
0,-0.000115,0,0.221913,0.00,0.000000,0.193225
1,-0.000426,0,0.128674,0.12,0.019852,0.045096
2,0.000232,0,0.113978,0.66,0.081491,0.021579
3,0.000182,0,0.147132,0.97,0.065019,0.015136
4,-0.000560,0,0.147132,0.47,0.047145,0.112034


<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Shape-based features capture the overall structure of the signal within each window, focusing on global patterns rather than local variations. 
The <b>trend slope</b> provides a continuous measure of the signal direction, while the <b>trend type</b> categorizes the window into increasing, decreasing, or stable regimes.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The <b>peak value</b> represents the maximum response within the window, and the <b>peak position</b> indicates where this response occurs in time. 
This helps differentiate early-response signals from delayed-response patterns, which can be characteristic of different gas conditions or sensor coatings.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The <b>rise strength</b> measures how strongly the signal increases from the start of the window to its peak, reflecting response intensity. 
Similarly, the <b>fall strength</b> captures how much the signal decreases after the peak, providing insight into recovery behavior.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Together, these features describe the overall response–recovery shape of the signal, which is particularly important in gas sensing applications where different gases produce distinct temporal response profiles.
</p>

<h2 style="color:#1B4F72; font-weight:bold;">
5. Variability Features
</h2>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Variability features extend the feature set with descriptors of signal energy, distribution shape, local stability, and oscillatory behavior. 
They are designed to capture aspects of the signal that are not fully represented by basic statistical summaries or first-order temporal changes. <i>(Literature Review, Section 3.2, Feature Engineering for Gas Sensor Time-Series)</i>
</p>

In [15]:
import numpy as np
from scipy.stats import kurtosis, skew

def extract_variability_features(signal):
    signal = np.asarray(signal)
    n = len(signal)

    if n < 2:
        return {
            "net_change": 0,
            "energy": 0,
            "skewness": 0,
            "kurtosis": 0,
            "flatness": 0,
            "zero_crossing_rate_diff": 0,
            "signal_start": 0,
            "signal_end": 0,
            "peak_to_start_distance": 0,
            "peak_to_end_distance": 0,
            "relative_peak_position": 0
        }

    diff = np.diff(signal)

    net_change = signal[-1] - signal[0]
    energy = np.mean(signal ** 2)  # normalized energy

    # robust shape statistics
    if np.std(signal) < 1e-8:
        skewness = 0
        kurt = 0
    else:
        skewness = skew(signal)
        kurt = kurtosis(signal)

    flatness_threshold = 1e-3
    flatness = np.mean(np.abs(diff) < flatness_threshold)

    diff_sign = np.sign(diff)
    zero_crossing_rate_diff = (
        np.sum(diff_sign[:-1] != diff_sign[1:]) / len(diff_sign[1:])
        if len(diff_sign) > 1 else 0
    )

    peak_position = int(np.argmax(signal))
    peak_to_start_distance = peak_position / (n - 1)
    peak_to_end_distance = (n - 1 - peak_position) / (n - 1)
    relative_peak_position = peak_position / (n - 1)

    return {
        "net_change": net_change,
        "energy": energy,
        "skewness": skewness,
        "kurtosis": kurt,
        "flatness": flatness,
        "zero_crossing_rate_diff": zero_crossing_rate_diff,
        "signal_start": signal[0],
        "signal_end": signal[-1],
        "peak_to_start_distance": peak_to_start_distance,
        "peak_to_end_distance": peak_to_end_distance,
        "relative_peak_position": relative_peak_position
    }

variability_features = windowed_df["signal_window"].apply(extract_variability_features)
variability_df = pd.DataFrame(variability_features.tolist())

display(variability_df.head())

,net_change,energy,skewness,kurtosis,flatness,zero_crossing_rate_diff,signal_start,signal_end,peak_to_start_distance,peak_to_end_distance,relative_peak_position
0,-0.193225,0.008947,0.415783,2.435652,0.151515,0.193878,0.221913,0.028688,0.000000,1.000000,0.000000
1,-0.025244,0.007669,-0.536299,-0.479818,0.171717,0.224490,0.108822,0.083578,0.121212,0.878788,0.121212
2,0.059912,0.006818,-0.703570,0.285166,0.171717,0.224490,0.032487,0.092399,0.666667,0.333333,0.666667
3,0.049883,0.009255,0.342366,-0.604739,0.131313,0.163265,0.082112,0.131995,0.979798,0.020202,0.979798
4,-0.064889,0.009037,-0.622299,-0.337232,0.090909,0.122449,0.099987,0.035098,0.474747,0.525253,0.474747


<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Variability features provide a broader characterization of each window by describing signal evolution, energy, distribution shape, local flatness, and the temporal location of the peak response.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The <b>net_change</b> feature summarizes the overall difference between the start and end of the window, helping distinguish rising, falling, and transition-dominated segments. 
The <b>energy</b> feature reflects the overall magnitude of the signal within the window, providing a compact measure of signal intensity.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The <b>skewness</b> and <b>kurtosis</b> features describe the shape of the signal distribution, helping differentiate balanced windows from those dominated by asymmetric or sharp responses. 
The <b>flatness</b> feature indicates the proportion of locally stable regions, while <b>zero_crossing_rate_diff</b> captures the frequency of directional alternation in short-term changes.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Peak-related features such as <b>peak_to_start_distance</b>, <b>peak_to_end_distance</b>, and <b>relative_peak_position</b> indicate where the strongest response occurs within the window. 
This helps distinguish windows centered on rising phases, peak regions, or recovery behavior.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Together, these features complement the statistical, temporal, and shape-based descriptors by capturing signal intensity, stability, oscillation, and phase position in a more detailed way.
</p>

<h2 style="color:#1B4F72; font-weight:bold;">
6. Feature Aggregation
</h2>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
All extracted feature groups are consolidated into a single structured table, combining statistical, temporal, shape-based, and variability descriptors with run-level and window-level metadata. 
This unified representation serves as the final feature matrix for downstream modeling and evaluation.
</p>

In [19]:
assert len(windowed_df) == len(stat_df) == len(temporal_df) == len(shape_df) == len(variability_df), \
    "Feature tables do not have matching row counts."

windowed_df = windowed_df.reset_index(drop=True)
stat_df = stat_df.reset_index(drop=True)
temporal_df = temporal_df.reset_index(drop=True)
shape_df = shape_df.reset_index(drop=True)
variability_df = variability_df.reset_index(drop=True)

features_df = pd.concat(
    [
        windowed_df[
            [
                "run_id",
                "experiment",
                "experiment_folder",
                "run_folder",
                "repeat_index",
                "window_id",
                "start_idx",
                "end_idx",
                "time_start",
                "time_end"
            ]
        ],
        stat_df,
        temporal_df,
        shape_df,
        variability_df
    ],
    axis=1
)

print("Combined feature table shape:", features_df.shape)
display(features_df.head())

Combined feature table shape: (2353, 43)


,run_id,experiment,experiment_folder,run_folder,repeat_index,window_id,start_idx,end_idx,time_start,time_end,...,energy,skewness,kurtosis,flatness,zero_crossing_rate_diff,signal_start,signal_end,peak_to_start_distance,peak_to_end_distance,relative_peak_position
0,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,NaN,Mixing_LowHigh_concentrations__UV ink-exposed ...,0,99,0.000,115.869,...,0.008947,0.415783,2.435652,0.151515,0.193878,0.221913,0.028688,0.000000,1.000000,0.000000
1,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,NaN,Mixing_LowHigh_concentrations__UV ink-exposed ...,50,149,58.252,175.526,...,0.007669,-0.536299,-0.479818,0.171717,0.224490,0.108822,0.083578,0.121212,0.878788,0.121212
2,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,NaN,Mixing_LowHigh_concentrations__UV ink-exposed ...,100,199,117.016,234.546,...,0.006818,-0.703570,0.285166,0.171717,0.224490,0.032487,0.092399,0.666667,0.333333,0.666667
3,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,NaN,Mixing_LowHigh_concentrations__UV ink-exposed ...,150,249,176.640,293.550,...,0.009255,0.342366,-0.604739,0.131313,0.163265,0.082112,0.131995,0.979798,0.020202,0.979798
4,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,NaN,Mixing_LowHigh_concentrations__UV ink-exposed ...,200,299,235.821,352.716,...,0.009037,-0.622299,-0.337232,0.090909,0.122449,0.099987,0.035098,0.474747,0.525253,0.474747


<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The resulting feature table combines all extracted descriptors with their corresponding metadata, ensuring that each window is represented by a comprehensive set of features. 
Each row corresponds to a single window, enriched with statistical summaries, temporal dynamics, shape characteristics, and variability measures.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Metadata fields such as <b>run_id</b>, <b>experiment</b>, and <b>window_id</b> preserve traceability, allowing features to be linked back to their original experimental context and time segment.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This unified structure enables efficient model training, evaluation, and analysis, while maintaining consistency with the run-aware design principles established in earlier stages of the pipeline.
</p>

<h2 style="color:#1B4F72; font-weight:bold;">
7. Final Modeling Dataset
</h2>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The final modeling dataset is prepared by separating predictive features from the target label while excluding metadata fields used only for traceability, grouping, and window identification.
This produces a clean feature matrix <b>X</b> and target vector <b>y</b> for downstream machine learning models.
</p>

In [21]:
target_col = "experiment"

metadata_cols = [
    "run_id",
    "experiment_folder",
    "run_folder",
    "repeat_index",
    "window_id",
    "start_idx",
    "end_idx",
    "time_start",
    "time_end"
]

feature_cols = [
    col for col in features_df.columns
    if col not in metadata_cols and col != target_col
]

X = features_df[feature_cols].copy()
y = features_df[target_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target classes:", y.nunique())
display(X.head())
display(y.head())

X shape: (2353, 33)
y shape: (2353,)
Target classes: 6


,mean,std,min,max,median,range,iqr,cv,first_diff_mean,first_diff_std,...,energy,skewness,kurtosis,flatness,zero_crossing_rate_diff,signal_start,signal_end,peak_to_start_distance,peak_to_end_distance,relative_peak_position
0,0.089142,0.031639,0.020605,0.221913,0.094835,0.201308,0.033844,0.354935,-0.001952,0.008036,...,0.008947,0.415783,2.435652,0.151515,0.193878,0.221913,0.028688,0.000000,1.000000,0.000000
1,0.082972,0.028013,0.020605,0.128674,0.089349,0.108069,0.035378,0.337617,-0.000255,0.004661,...,0.007669,-0.536299,-0.479818,0.171717,0.224490,0.108822,0.083578,0.121212,0.878788,0.121212
2,0.080224,0.019554,0.030791,0.113978,0.082813,0.083187,0.023622,0.243743,0.000605,0.004608,...,0.006818,-0.703570,0.285166,0.171717,0.224490,0.032487,0.092399,0.666667,0.333333,0.666667
3,0.093657,0.021975,0.057616,0.147132,0.090766,0.089516,0.033182,0.234636,0.000504,0.004457,...,0.009255,0.342366,-0.604739,0.131313,0.163265,0.082112,0.131995,0.979798,0.020202,0.979798
4,0.088229,0.035386,0.008150,0.147132,0.091891,0.138982,0.048667,0.401074,-0.000655,0.005779,...,0.009037,-0.622299,-0.337232,0.090909,0.122449,0.099987,0.035098,0.474747,0.525253,0.474747


0    Mixing_LowHigh_concentrations
1    Mixing_LowHigh_concentrations
2    Mixing_LowHigh_concentrations
3    Mixing_LowHigh_concentrations
4    Mixing_LowHigh_concentrations
Name: experiment, dtype: str

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The resulting feature matrix contains 2353 window-level samples described by 33 engineered features, while the target vector preserves the six experiment classes used for classification.
This confirms that the feature engineering stage has successfully transformed the original windowed signals into a structured machine-learning dataset.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Metadata fields such as run identifiers, folder labels, and temporal indices are intentionally excluded from <b>X</b> because they do not represent intrinsic signal behavior and could introduce artificial shortcuts during model training.
Keeping these fields outside the predictor matrix preserves the validity of downstream evaluation.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
At this point, the dataset is ready for model-specific preparation steps such as train-test partitioning, label encoding if required, and baseline model training.
</p>

<h3 style="color:#2874A6; font-weight:bold;">
Dataset Export
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The finalized feature table is validated and exported for downstream modeling and reproducible reuse within the pipeline. 
Metadata columns are retained for traceability, while engineered features form the predictive input space.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The dataset is stored in Parquet format to ensure efficient storage, fast loading, and compatibility with scalable data processing workflows.
</p>

In [31]:
from pathlib import Path
import warnings
from pandas.errors import Pandas4Warning

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=Pandas4Warning)

# ----------------------------
# 1) Define metadata columns
# ----------------------------
metadata_cols = [
    "run_id",
    "experiment",
    "experiment_folder",
    "run_folder",
    "repeat_index",
    "window_id",
    "start_idx",
    "end_idx",
    "time_start",
    "time_end"
]

# ----------------------------
# 2) Reorder columns
# ----------------------------
feature_cols = [col for col in features_df.columns if col not in metadata_cols]
features_df = features_df[metadata_cols + feature_cols]

# ----------------------------
# 3) Data validation
# ----------------------------
print("Final shape:", features_df.shape)

missing_total = features_df.isnull().sum().sum()
print("Total missing values:", missing_total)

# safer than select_dtypes(include="object")
object_cols = features_df.columns[features_df.dtypes.eq("object")].tolist()
print("Object columns:", object_cols)

if "trend_type" in features_df.columns:
    print("Trend type unique values:", features_df["trend_type"].unique())
else:
    print("Column 'trend_type' not found.")

display(features_df.head())

# Optional summary of engineered features only
engineered_feature_cols = [col for col in features_df.columns if col not in metadata_cols]
display(features_df[engineered_feature_cols].describe().T.head(10))

# ----------------------------
# Handle missing values
# ----------------------------
from sklearn.impute import SimpleImputer

feature_cols = [col for col in features_df.columns if col not in metadata_cols]

# Fix feature NaNs
imputer = SimpleImputer(strategy="median")
features_df[feature_cols] = imputer.fit_transform(features_df[feature_cols])

# Fix metadata NaNs (example)
features_df["repeat_index"] = features_df["repeat_index"].fillna(-1)

print("Remaining missing values:", features_df.isnull().sum().sum())

# ----------------------------
# 4) Save dataset
# ----------------------------
OUTPUT_PATH = Path("../data/processed")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

output_file = OUTPUT_PATH / "features_timeseries.parquet"
features_df.to_parquet(output_file, index=False)

print(f"\nSaved to: {output_file}")

Final shape: (2353, 43)
Total missing values: 0
Object columns: []
Trend type unique values: [ 0.  1. -1.]


,run_id,experiment,experiment_folder,run_folder,repeat_index,window_id,start_idx,end_idx,time_start,time_end,...,energy,skewness,kurtosis,flatness,zero_crossing_rate_diff,signal_start,signal_end,peak_to_start_distance,peak_to_end_distance,relative_peak_position
0,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,-1.0,Mixing_LowHigh_concentrations__UV ink-exposed ...,0,99,0.000,115.869,...,0.008947,0.415783,2.435652,0.151515,0.193878,0.221913,0.028688,0.000000,1.000000,0.000000
1,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,-1.0,Mixing_LowHigh_concentrations__UV ink-exposed ...,50,149,58.252,175.526,...,0.007669,-0.536299,-0.479818,0.171717,0.224490,0.108822,0.083578,0.121212,0.878788,0.121212
2,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,-1.0,Mixing_LowHigh_concentrations__UV ink-exposed ...,100,199,117.016,234.546,...,0.006818,-0.703570,0.285166,0.171717,0.224490,0.032487,0.092399,0.666667,0.333333,0.666667
3,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,-1.0,Mixing_LowHigh_concentrations__UV ink-exposed ...,150,249,176.640,293.550,...,0.009255,0.342366,-0.604739,0.131313,0.163265,0.082112,0.131995,0.979798,0.020202,0.979798
4,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,-1.0,Mixing_LowHigh_concentrations__UV ink-exposed ...,200,299,235.821,352.716,...,0.009037,-0.622299,-0.337232,0.090909,0.122449,0.099987,0.035098,0.474747,0.525253,0.474747


,count,mean,std,min,25%,50%,75%,max
mean,2353.0,0.321082,0.242752,0.031783,0.088644,2.523889e-01,0.502703,0.966298
std,2353.0,0.059449,0.068765,0.001122,0.015737,3.505045e-02,0.075556,0.391568
min,2353.0,0.216676,0.220745,-0.004536,0.054260,1.124969e-01,0.321538,0.936729
max,2353.0,0.428843,0.288459,0.043292,0.136582,4.003128e-01,0.683225,0.999208
median,2353.0,0.319641,0.253216,0.032167,0.082948,2.135055e-01,0.514693,0.966978
range,2353.0,0.212167,0.205622,0.004651,0.060912,1.341038e-01,0.302604,0.946158
iqr,2353.0,0.085644,0.122670,0.000744,0.020014,4.725695e-02,0.097527,0.834753
cv,2353.0,0.215719,0.214092,0.007568,0.088691,1.498444e-01,0.263649,1.529676
first_diff_mean,2353.0,0.000030,0.002181,-0.009238,-0.000411,-5.393908e-07,0.000442,0.009071
first_diff_std,2353.0,0.017349,0.021278,0.000488,0.003869,9.243997e-03,0.022451,0.116377


Remaining missing values: 0

Saved to: ..\data\processed\features_timeseries.parquet


<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The finalized feature table contains 2353 window-level samples described by 43 columns, including both metadata and engineered features. The dataset structure is consistent, and all expected feature groups are successfully integrated into a single representation.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
All engineered features are numerical, ensuring compatibility with machine learning models. Any object-type columns that remain are strictly part of the metadata and are not used as predictive inputs during modeling.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Missing values introduced during feature extraction were handled prior to dataset export. Feature columns were imputed using a median-based strategy to ensure robustness, while missing values in metadata fields (e.g., repeat_index) were replaced with a placeholder value to preserve dataset consistency without affecting model inputs. <i>(Literature Review, Section 2, Data Preprocessing for Gas Sensor Time-Series)</i>
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, the dataset is clean, well-structured, and fully numerical in its predictive feature space, making it ready for downstream modeling. Saving the data in Parquet format ensures efficient storage, reproducibility, and seamless integration into subsequent modeling stages.
</p>